# DINOv2 水体问题识别 - 完整训练与评估

**功能**：
- ✅ 完整训练流程（从数据加载到模型保存）
- ✅ 每轮训练指标记录（损失、准确率）
- ✅ **每个类别的准确率分析**
- ✅ 混淆矩阵可视化
- ✅ 训练曲线绘制

**优化策略**：
- 冻结前 20 层，只训练最后 2 层
- 混合精度训练 (FP16)
- 梯度累积
- 梯度检查点

**硬件要求**：NVIDIA GPU (建议 8GB+ 显存)

## 1. 环境检查与依赖安装

In [ ]:
# 检查依赖
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import json

print(f"Python 版本：{sys.version}")
print(f"PyTorch 版本：{torch.__version__}")
print(f"CUDA 可用：{torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# 安装缺失的依赖（如果需要）
!pip install transformers scikit-learn matplotlib pillow -q

## 2. 配置参数

In [ ]:
# 配置参数
CONFIG = {
    # 数据配置
    'data_root': r'H:\code\01_image_center\data\images\raw',
    'val_split': 0.2,
    'image_size': 518,
    
    # 模型配置
    'model_name': 'dinov2-large',  # dinov2-small / dinov2-base / dinov2-large
    'frozen_layers': 20,  # 冻结前 20 层
    
    # 训练配置
    'epochs': 20,
    'batch_size': 1,
    'accumulation_steps': 8,  # 梯度累积
    'learning_rate': 1e-5,
    'weight_decay': 0.05,
    'num_workers': 0,  # Windows 建议设为 0
    
    # 输出配置
    'output_dir': r'H:\code\01_image_center\runs\finetune',
    'experiment_name': 'exp1'
}

print("配置参数:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 3. 数据加载

In [ ]:
from sklearn.model_selection import train_test_split
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

class WaterProblemDataset(Dataset):
    """水利问题数据集"""
    
    def __init__(self, image_paths, labels, image_size=518, is_training=True):
        self.image_paths = image_paths
        self.labels = labels
        self.image_size = image_size
        self.is_training = is_training
        
        # 数据增强（仅训练集）
        if is_training:
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225]),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225]),
            ])
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
            tensor = self.transform(image)
            return tensor, label
        except Exception as e:
            print(f"[ERROR] 读取图像失败 {img_path}: {e}")
            return torch.zeros(3, self.image_size, self.image_size), -1

def load_data(data_root, val_split=0.2):
    """加载数据并划分训练/验证集"""
    print(f"\n[1] 加载数据集...")
    
    data_root = Path(data_root)
    image_paths = []
    labels = []
    label_names = []
    
    # 按子文件夹分类
    for class_dir in sorted(data_root.iterdir()):
        if class_dir.is_dir():
            class_name = class_dir.name
            label_names.append(class_name)
            
            for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
                for img_file in class_dir.glob(f'*{ext}'):
                    image_paths.append(str(img_file))
                    labels.append(len(label_names) - 1)
    
    print(f"    总图像数：{len(image_paths)}")
    print(f"    类别数：{len(label_names)}")
    print(f"    类别：{label_names}")
    
    # 统计每个类别的样本数
    print(f"\n    各类别样本数:")
    for i, name in enumerate(label_names):
        count = labels.count(i)
        print(f"      {name}: {count} 张")
    
    # 划分训练集和验证集
    if len(image_paths) > 0:
        train_paths, val_paths, train_labels, val_labels = train_test_split(
            image_paths, labels,
            test_size=val_split,
            random_state=42,
            stratify=labels
        )
        
        print(f"\n    训练集：{len(train_paths)} 张")
        print(f"    验证集：{len(val_paths)} 张")
    else:
        train_paths, val_paths = [], []
        train_labels, val_labels = [], []
    
    return (train_paths, train_labels), (val_paths, val_labels), label_names

# 加载数据
(train_paths, train_labels), (val_paths, val_labels), label_names = load_data(
    CONFIG['data_root'], CONFIG['val_split']
)

print(f"\n类别名称：{label_names}")

## 4. 创建数据加载器

In [ ]:
# 创建数据集
train_dataset = WaterProblemDataset(train_paths, train_labels, 
                                     CONFIG['image_size'], is_training=True)
val_dataset = WaterProblemDataset(val_paths, val_labels, 
                                   CONFIG['image_size'], is_training=False)

# 创建数据加载器
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True
)

print(f"训练批次数：{len(train_loader)}")
print(f"验证批次数：{len(val_loader)}")

## 5. 加载 DINOv2 模型

In [ ]:
from transformers import Dinov2Model

def load_dinov2_for_finetuning(model_name='dinov2-large', frozen_layers=20):
    """加载 DINOv2 模型进行微调"""
    print(f"\n[2] 加载 DINOv2 模型...")
    
    model_mapping = {
        'dinov2-small': 'facebook/dinov2-small',
        'dinov2-base': 'facebook/dinov2-base',
        'dinov2-large': 'facebook/dinov2-large'
    }
    
    model_path = model_mapping.get(model_name, 'facebook/dinov2-large')
    print(f"    模型：{model_name}")
    print(f"    路径：{model_path}")
    
    # 加载模型
    model = Dinov2Model.from_pretrained(model_path)
    
    # 冻结前 N 层
    total_layers = len(model.encoder.layer)
    print(f"    总层数：{total_layers}")
    print(f"    冻结层数：{frozen_layers}")
    print(f"    训练层数：{total_layers - frozen_layers}")
    
    for i, layer in enumerate(model.encoder.layer):
        if i < frozen_layers:
            for param in layer.parameters():
                param.requires_grad = False
        else:
            for param in layer.parameters():
                param.requires_grad = True
    
    # 启用梯度检查点
    model.gradient_checkpointing = True
    
    # 移动到 GPU
    if torch.cuda.is_available():
        model = model.cuda()
    
    print(f"    [OK] 模型加载成功")
    
    return model, total_layers - frozen_layers

# 加载模型
model, trainable_layers = load_dinov2_for_finetuning(
    CONFIG['model_name'], CONFIG['frozen_layers']
)

## 6. 创建检测头

In [ ]:
import torch.nn as nn

def create_detection_head(embed_dim, num_classes):
    """创建分类头"""
    print(f"\n[3] 创建检测头...")
    
    head = nn.Sequential(
        nn.Linear(embed_dim, 512),
        nn.GELU(),
        nn.Dropout(0.1),
        nn.Linear(512, num_classes)
    )
    
    # 初始化权重
    for m in head.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    
    print(f"    输入维度：{embed_dim}")
    print(f"    类别数：{num_classes}")
    print(f"    [OK] 检测头创建成功")
    
    return head

# 创建检测头
embed_dim = model.config.hidden_size
num_classes = len(label_names)
head = create_detection_head(embed_dim, num_classes)

if torch.cuda.is_available():
    head = head.cuda()

print(f"\n检测头结构:")
print(head)

## 7. 设置优化器和损失函数

In [ ]:
from torch.cuda.amp import autocast, GradScaler

# 优化器
optimizer = torch.optim.AdamW([
    {'params': model.parameters(), 'lr': CONFIG['learning_rate']},
    {'params': head.parameters(), 'lr': CONFIG['learning_rate'] * 10}
], weight_decay=CONFIG['weight_decay'])

# 损失函数
criterion = nn.CrossEntropyLoss()

# 混合精度 scaler
scaler = GradScaler()

print("[OK] 优化器和损失函数已设置")
print(f"  优化器：AdamW")
print(f"  基础学习率：{CONFIG['learning_rate']}")
print(f"  检测头学习率：{CONFIG['learning_rate'] * 10}")

## 8. 训练与验证函数（含分类报告）

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def train_one_epoch(model, head, dataloader, optimizer, criterion, scaler, 
                   accumulation_steps, epoch):
    """训练一个 epoch"""
    model.train()
    head.train()
    
    total_loss = 0
    correct = 0
    total = 0
    
    optimizer.zero_grad()
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        if labels[0] == -1:
            continue
        
        if torch.cuda.is_available():
            images = images.cuda()
            labels = labels.cuda()
        
        # 混合精度训练
        with autocast():
            outputs = model(images)
            features = outputs.last_hidden_state[:, 0, :]
            logits = head(features)
            loss = criterion(logits, labels) / accumulation_steps
        
        scaler.scale(loss).backward()
        
        if (batch_idx + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss += loss.item() * accumulation_steps
        _, predicted = logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if (batch_idx + 1) % 10 == 0:
            progress = (batch_idx + 1) / len(dataloader) * 100
            acc = 100. * correct / total
            print(f"    Epoch {epoch} [{progress:.1f}%] Loss: {total_loss/(batch_idx+1):.4f} Acc: {acc:.2f}%")
        
        del images, labels, outputs, features, logits, loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return total_loss / len(dataloader), 100. * correct / total

def validate_with_per_class(model, head, dataloader, criterion, label_names):
    """验证并计算每个类别的准确率"""
    model.eval()
    head.eval()
    
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            if labels[0] == -1:
                continue
            
            if torch.cuda.is_available():
                images = images.cuda()
                labels = labels.cuda()
            
            outputs = model(images)
            features = outputs.last_hidden_state[:, 0, :]
            logits = head(features)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            _, predicted = logits.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            del images, labels, outputs, features, logits, loss
    
    # 整体准确率
    overall_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels) * 100
    
    # 每个类别的准确率
    per_class_acc = {}
    for i, name in enumerate(label_names):
        class_indices = [j for j, l in enumerate(all_labels) if l == i]
        if len(class_indices) > 0:
            correct = sum(1 for j in class_indices if all_preds[j] == i)
            per_class_acc[name] = correct / len(class_indices) * 100
        else:
            per_class_acc[name] = 0.0
    
    return (total_loss / len(dataloader), 
            overall_acc, 
            per_class_acc,
            np.array(all_labels),
            np.array(all_preds))

print("[OK] 训练和验证函数已定义")

## 9. 开始训练

In [ ]:
import time

# 训练记录
training_history = []
best_acc = 0
best_per_class_acc = None

print("=" * 80)
print("开始训练")
print("=" * 80)
print(f"时间：{datetime.now()}")
print(f" epochs: {CONFIG['epochs']}")
print(f" 批次大小：{CONFIG['batch_size']}")
print(f" 梯度累积：{CONFIG['accumulation_steps']}")
print(f" 学习率：{CONFIG['learning_rate']}")
print(f"=" * 80)

for epoch in range(1, CONFIG['epochs'] + 1):
    epoch_start = time.time()
    
    print(f"\n{'='*80}")
    print(f"Epoch {epoch}/{CONFIG['epochs']}")
    print(f"{'='*80}")
    
    # 训练
    train_loss, train_acc = train_one_epoch(
        model, head, train_loader, optimizer, criterion, scaler,
        CONFIG['accumulation_steps'], epoch
    )
    
    # 验证
    val_loss, val_acc, per_class_acc, val_labels, val_preds = validate_with_per_class(
        model, head, val_loader, criterion, label_names
    )
    
    epoch_time = (time.time() - epoch_start) / 60
    
    print(f"\n[结果] Epoch {epoch} (用时：{epoch_time:.1f} 分钟)")
    print(f"  训练损失：{train_loss:.4f}")
    print(f"  训练准确率：{train_acc:.2f}%")
    print(f"  验证损失：{val_loss:.4f}")
    print(f"  验证准确率：{val_acc:.2f}%")
    
    print(f"\n  各类别准确率:")
    for name, acc in per_class_acc.items():
        print(f"    {name}: {acc:.1f}%")
    
    # 记录历史
    training_history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'per_class_acc': per_class_acc,
        'time': epoch_time
    })
    
    # 保存最佳模型
    if val_acc > best_acc:
        best_acc = val_acc
        best_per_class_acc = per_class_acc.copy()
        print(f"\n  [OK] 新最佳模型！准确率：{best_acc:.2f}%")
        
        output_dir = Path(CONFIG['output_dir'])
        output_dir.mkdir(parents=True, exist_ok=True)
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'head_state_dict': head.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'per_class_acc': per_class_acc,
            'label_names': label_names
        }, output_dir / 'best_model.pth')
        
        print(f"  [OK] 模型已保存：{output_dir / 'best_model.pth'}")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*80}")
print("训练完成!")
print(f"{'='*80}")
print(f"最佳验证准确率：{best_acc:.2f}%")
print(f"最佳模型各类别准确率:")
if best_per_class_acc:
    for name, acc in best_per_class_acc.items():
        print(f"  {name}: {acc:.1f}%")

## 10. 保存训练历史

In [ ]:
# 保存训练历史为 JSON
history_file = Path(CONFIG['output_dir']) / 'training_log.json'
history_file.parent.mkdir(parents=True, exist_ok=True)

# 转换 per_class_acc 为可序列化格式
history_serializable = []
for record in training_history:
    record_copy = record.copy()
    record_copy['per_class_acc'] = {k: float(v) for k, v in record['per_class_acc'].items()}
    history_serializable.append(record_copy)

with open(history_file, 'w', encoding='utf-8') as f:
    json.dump(history_serializable, f, indent=2, ensure_ascii=False)

print(f"训练历史已保存：{history_file}")

## 11. 绘制训练曲线

In [ ]:
# 绘制训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 损失曲线
axes[0].plot([h['epoch'] for h in training_history], 
             [h['train_loss'] for h in training_history], 
             'b-', label='训练损失', linewidth=2)
axes[0].plot([h['epoch'] for h in training_history], 
             [h['val_loss'] for h in training_history], 
             'r--', label='验证损失', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('训练损失 vs 验证损失', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 准确率曲线
axes[1].plot([h['epoch'] for h in training_history], 
             [h['train_acc'] for h in training_history], 
             'b-', label='训练准确率', linewidth=2)
axes[1].plot([h['epoch'] for h in training_history], 
             [h['val_acc'] for h in training_history], 
             'r--', label='验证准确率', linewidth=2)
axes[1].axhline(y=best_acc, color='g', linestyle=':', label=f'最佳验证准确率 ({best_acc:.1f}%)')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('训练准确率 vs 验证准确率', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

# 保存图表
plot_file = Path(CONFIG['output_dir']) / 'training_curves.png'
plt.savefig(plot_file, dpi=150, bbox_inches='tight')
print(f"训练曲线已保存：{plot_file}")

plt.show()

## 12. 绘制各类别准确率对比

In [ ]:
# 绘制各类别准确率对比（最佳模型）
if best_per_class_acc:
    plt.figure(figsize=(10, 6))
    
    names = list(best_per_class_acc.keys())
    accs = list(best_per_class_acc.values())
    
    colors = ['#2ecc71' if acc >= 70 else '#f39c12' if acc >= 50 else '#e74c3c' 
              for acc in accs]
    
    bars = plt.bar(names, accs, color=colors, edgecolor='black', linewidth=1.5)
    
    plt.xlabel('类别', fontsize=12)
    plt.ylabel('准确率 (%)', fontsize=12)
    plt.title('最佳模型 - 各类别验证准确率', fontsize=14)
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 100)
    
    # 添加数值标签
    for bar, acc in zip(bars, accs):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                f'{acc:.1f}%', ha='center', va='bottom', fontsize=10)
    
    plt.axhline(y=70, color='green', linestyle='--', alpha=0.5, label='良好 (70%)')
    plt.axhline(y=50, color='orange', linestyle='--', alpha=0.5, label='一般 (50%)')
    plt.legend()
    
    plt.tight_layout()
    
    # 保存图表
    class_acc_file = Path(CONFIG['output_dir']) / 'per_class_accuracy.png'
    plt.savefig(class_acc_file, dpi=150, bbox_inches='tight')
    print(f"各类别准确率图已保存：{class_acc_file}")
    
    plt.show()

## 13. 绘制混淆矩阵

In [ ]:
import seaborn as sns

# 使用最后一次验证的结果绘制混淆矩阵
cm = confusion_matrix(val_labels, val_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('预测标签', fontsize=12)
plt.ylabel('真实标签', fontsize=12)
plt.title('混淆矩阵', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

# 保存图表
cm_file = Path(CONFIG['output_dir']) / 'confusion_matrix.png'
plt.savefig(cm_file, dpi=150, bbox_inches='tight')
print(f"混淆矩阵已保存：{cm_file}")

plt.show()

## 14. 分类报告

In [ ]:
from sklearn.metrics import classification_report

# 生成分类报告
report = classification_report(val_labels, val_preds, 
                                target_names=label_names,
                                digits=4)

print("=" * 80)
print("分类报告")
print("=" * 80)
print(report)

# 保存报告
report_file = Path(CONFIG['output_dir']) / 'classification_report.txt'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write("分类报告\n")
    f.write("=" * 80 + "\n\n")
    f.write(report)

print(f"\n分类报告已保存：{report_file}")

## 15. 训练总结

In [ ]:
print("=" * 80)
print("训练总结")
print("=" * 80)

print(f"\n📊 整体性能:")
print(f"  最佳验证准确率：{best_acc:.2f}%")
print(f"  训练轮数：{CONFIG['epochs']}")
print(f"  总用时：{sum(h['time'] for h in training_history):.1f} 分钟")

print(f"\n📈 各类别性能:")
if best_per_class_acc:
    sorted_acc = sorted(best_per_class_acc.items(), key=lambda x: x[1], reverse=True)
    for name, acc in sorted_acc:
        status = "✅" if acc >= 70 else "⚠️" if acc >= 50 else "❌"
        print(f"  {status} {name}: {acc:.1f}%")

print(f"\n💾 输出文件:")
print(f"  最佳模型：{CONFIG['output_dir']}/best_model.pth")
print(f"  训练历史：{CONFIG['output_dir']}/training_log.json")
print(f"  训练曲线：{CONFIG['output_dir']}/training_curves.png")
print(f"  各类别准确率：{CONFIG['output_dir']}/per_class_accuracy.png")
print(f"  混淆矩阵：{CONFIG['output_dir']}/confusion_matrix.png")
print(f"  分类报告：{CONFIG['output_dir']}/classification_report.txt")

print(f"\n{'='*80}")
if best_acc >= 85:
    print("✅ 优秀！验证准确率 >= 85%")
    print("   建议：继续使用 A6000 进行完整训练")
elif best_acc >= 75:
    print("⭕ 良好！验证准确率 >= 75%")
    print("   建议：可以继续使用 A6000 训练，或调整超参数")
else:
    print("❌ 一般！验证准确率 < 75%")
    print("   建议：检查数据质量，增加数据量，调整超参数")
print(f"{'='*80}")